In [30]:
import re
import pandas as pd
import json
from datetime import datetime, timezone

In [31]:
with open("../data/rawpayload.json", "r") as fp:
    data = json.load(fp)

In [32]:
list_of_pages = data.get("pages")

In [ ]:
def _parse_page(page):
    pass

first_page = list_of_pages[0]

In [34]:
DATE_COLS = [
    "extracted_at",
    "created_at",
    "updated_at"
]

def _get_reviewers(reviewer_list: list[dict]) -> list[str]:
    if not reviewer_list:
        return []
    return [reviewer.get("login") for reviewer in reviewer_list if reviewer]

def _get_milestone(title: str, labels: list[dict]) -> str | None:
    milestone = None
    if labels:
        for label in labels:
            if label.get("name", "").startswith("M") and label.get("name", "")[1:].isdigit():
                milestone = label.get("name")
                break

    if milestone is None and title:
        match = re.search(r"\[(M\d+)\]", title)
        if match:
            milestone = match.group(1)

    return milestone

def _get_status(field_values: list[dict]) -> str:
    for field in field_values:
        if not field:
            continue
        if field.get("field", {}).get("name") == "Status":
            status = field.get("name")
            break
    return status

def _standardize_date_cols(table: pd.DataFrame, date_cols: list[str])-> pd.DataFrame:
    for date_col in date_cols:
        table[date_col] = (
            pd.to_datetime(table[date_col], utc=True)
            .dt.tz_convert("Asia/Manila")
            .dt.floor("s")
        )
    return table



In [35]:
extracted_at = datetime.now(timezone.utc)

list_of_contents = first_page["data"].get("organization").get("projectV2").get("items").get("nodes")
     
silver_df = pd.json_normalize(list_of_contents, sep="_")
silver_df.rename(
    columns={
        "content_id": "issue_id",
        "content_title": "issue_title",
        "content_url": "issue_url",
        "content_author_login": "issue_author",
        "content_createdAt": "created_at",
        "content_updatedAt": "updated_at",
        "content_state": "state",
        "content_assignees_nodes": "reviewer",
        "content_labels_nodes": "milestone",
        "fieldValues_nodes": "status"
    },
    inplace=True
)
silver_df["extracted_at"] = extracted_at
    
silver_df["reviewer"] = silver_df["reviewer"].apply(_get_reviewers)
silver_df["milestone"] = silver_df.apply(lambda row: _get_milestone(row["issue_title"], row["milestone"]), axis=1)
silver_df["status"] = silver_df["status"].apply(_get_status)

silver_df = _standardize_date_cols(silver_df, DATE_COLS)

# Feature engineer measure columns
silver_df["is_assigned"] = (silver_df["reviewer"].notna().astype("int8"))
silver_df["days_since_update"] = (silver_df["updated_at"] - silver_df["created_at"]).dt.days
silver_df["submission_age_days"] = (silver_df["extracted_at"] - silver_df["created_at"]).dt.days

silver = silver_df[silver_df["issue_title"].str.match(r"\[M\d+\]")]


In [36]:
silver.head()

,issue_id,issue_title,issue_url,issue_author,created_at,updated_at,state,reviewer,milestone,status,extracted_at,is_assigned,days_since_update,submission_age_days
0,I_kwDOShte388AAAABIkl5GA,[M2] <Froncoyz Verano> — <Philippine Regional ...,https://github.com/dataengineeringpilipinas/de...,Froncoyz,2026-07-13 11:50:03+08:00,2026-08-03 21:18:12+08:00,OPEN,[datadonotlieee],M2,Needs Improvement,2026-08-12 21:14:43+08:00,1,21,30
1,I_kwDOShte388AAAABK9ne-w,[M2] <Naehum Dela Cruz> — <MassKara Festival G...,https://github.com/dataengineeringpilipinas/de...,Naehum-Dela-Cruz,2026-07-31 22:46:52+08:00,2026-08-04 18:26:17+08:00,CLOSED,[jlnrox],M2,Passed,2026-08-12 21:14:43+08:00,1,3,11
2,I_kwDOShte388AAAABKgau4A,[M2] John Lopez — MaxiPay PH,https://github.com/dataengineeringpilipinas/de...,jatjatlopez,2026-07-28 23:06:08+08:00,2026-08-03 21:19:56+08:00,OPEN,[datadonotlieee],M2,Needs Improvement,2026-08-12 21:14:43+08:00,1,5,14
3,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-23 00:17:50+08:00,2026-08-02 09:58:56+08:00,OPEN,[webzero13],M0,Passed,2026-08-12 21:14:43+08:00,1,40,50
4,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10 20:18:57+08:00,2026-08-03 20:05:57+08:00,CLOSED,[nicholailim],M1,Passed,2026-08-12 21:14:43+08:00,1,23,33


In [ ]:
import numpy as np



In [38]:
silver.head()

,issue_id,issue_title,issue_url,issue_author,created_at,updated_at,state,reviewer,milestone,status,extracted_at,is_assigned,days_since_update,submission_age_days,current_milestone
0,I_kwDOShte388AAAABIkl5GA,[M2] <Froncoyz Verano> — <Philippine Regional ...,https://github.com/dataengineeringpilipinas/de...,Froncoyz,2026-07-13 11:50:03+08:00,2026-08-03 21:18:12+08:00,OPEN,[datadonotlieee],M2,Needs Improvement,2026-08-12 21:14:43+08:00,1,21,30,M2
1,I_kwDOShte388AAAABK9ne-w,[M2] <Naehum Dela Cruz> — <MassKara Festival G...,https://github.com/dataengineeringpilipinas/de...,Naehum-Dela-Cruz,2026-07-31 22:46:52+08:00,2026-08-04 18:26:17+08:00,CLOSED,[jlnrox],M2,Passed,2026-08-12 21:14:43+08:00,1,3,11,M3
2,I_kwDOShte388AAAABKgau4A,[M2] John Lopez — MaxiPay PH,https://github.com/dataengineeringpilipinas/de...,jatjatlopez,2026-07-28 23:06:08+08:00,2026-08-03 21:19:56+08:00,OPEN,[datadonotlieee],M2,Needs Improvement,2026-08-12 21:14:43+08:00,1,5,14,M2
3,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-23 00:17:50+08:00,2026-08-02 09:58:56+08:00,OPEN,[webzero13],M0,Passed,2026-08-12 21:14:43+08:00,1,40,50,M1
4,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10 20:18:57+08:00,2026-08-03 20:05:57+08:00,CLOSED,[nicholailim],M1,Passed,2026-08-12 21:14:43+08:00,1,23,33,M2


In [40]:
silver_df["issue_author"].unique()

<StringArray>
[              'Froncoyz',       'Naehum-Dela-Cruz',            'jatjatlopez',
              'penOnFire',             'navikram03',       'johnkennethpriol',
                  'xeind',              'jffabrero',              'kathulhur',
                'MCabs28',                'supjkay',                'ksdizon',
 'jimwellbryllsantos-dot',      'anonggentlegentle',             'shivaa-bit',
             'miksss1412',           'andreapongco',               'Zeraphim',
           'CardinalSeen',           'GeraldAlviar',                 'ssuish',
               'vn0raven',               'robaaaru',     'reinerjohnsantiago',
              'Kaigosama',                 'avsguy',                'neyu007',
            'DioVale2002',            'AdrianeJone',                'Sheyxii',
                'Jan-Ric',              'yugiakame',              'Monferium',
               'rainer-r',             'cancinoray',                 'abgacs',
                'lordgpm',            

In [43]:
import re
import numpy as np
import pandas as pd
from datetime import datetime

DATE_COLS = [
    "extracted_at",
    "created_at",
    "updated_at"
]

NAMES_TO_DROP = [
    "smmariquit", 
    "gkate78", 
    "Zeraphim", 
    "cancinoray", 
    "CardinalSeen"
]

def _get_reviewers(reviewer_list: list[dict]) -> list[str]:
    if not reviewer_list:
        return []
    return [reviewer.get("login") for reviewer in reviewer_list if reviewer]

def _get_milestone(title: str, labels: list[dict]) -> str | None:
    milestone = None
    if labels:
        for label in labels:
            if label.get("name", "").startswith("M") and label.get("name", "")[1:].isdigit():
                milestone = label.get("name")
                break

    if milestone is None and title:
        match = re.search(r"\[(M\d+)\]", title)
        if match:
            milestone = match.group(1)

    return milestone

def _get_status(field_values: list[dict]) -> str | None:
    status = None
    for field in field_values:
        if not field:
            continue
        if field.get("field", {}).get("name") == "Status":
            status = field.get("name")
            break
    return status

def _standardize_date_cols(table: pd.DataFrame, date_cols: list[str])-> pd.DataFrame:
    for date_col in date_cols:
        table[date_col] = (
            pd.to_datetime(table[date_col], utc=True)
            .dt.tz_convert("Asia/Manila")
            .dt.floor("s")
        )
    return table

def _transform_page_to_silver(
    run_id: str,
    extracted_at: datetime,
    page: dict,
) -> pd.DataFrame:

    list_of_contents = (
        page["data"]
        .get("organization")
        .get("projectV2")
        .get("items")
        .get("nodes")
    )
     
    silver_df = pd.json_normalize(list_of_contents, sep="_")
    silver_df.rename(
        columns={
            "content_id": "issue_id",
            "content_title": "issue_title",
            "content_url": "issue_url",
            "content_author_login": "issue_author",
            "content_createdAt": "created_at",
            "content_updatedAt": "updated_at",
            "content_state": "state",
            "content_assignees_nodes": "reviewer",
            "content_labels_nodes": "milestone",
            "fieldValues_nodes": "status"
        },
        inplace=True
    )
    
    silver_df["run_id"] = run_id
    silver_df["extracted_at"] = extracted_at
    
    silver_df["reviewer"] = silver_df["reviewer"].apply(_get_reviewers)
    silver_df["milestone"] = silver_df.apply(lambda row: _get_milestone(row["issue_title"], row["milestone"]), axis=1)
    silver_df["status"] = silver_df["status"].apply(_get_status)
    silver_df = silver_df[~silver_df["issue_author"].isin(NAMES_TO_DROP)]
    silver_df = silver_df[
        silver_df["issue_title"].str.match(r"\[M\d+\]", na=False)
    ].copy()

    silver_df = _standardize_date_cols(silver_df, DATE_COLS)
    
    # Feature engineer measure columns
    silver_df["is_assigned"] = (silver_df["reviewer"].notna().astype("int8"))
    silver_df["days_since_update"] = (silver_df["updated_at"] - silver_df["created_at"]).dt.days
    silver_df["submission_age_days"] = (silver_df["extracted_at"] - silver_df["created_at"]).dt.days
    
    milestone_num = silver_df["milestone"].str.extract(r"(\d+)")[0].astype(int)

    silver_df["current_milestone"] = np.where(
        silver_df["status"].str.lower().eq("passed"),
        "M" + (milestone_num + 1).clip(upper=6).astype(str),
        silver_df["milestone"]
    )
    return silver_df


def transform_bronze_to_silver(
    run_id: str,
    extracted_at: datetime,
    payload: dict,
) -> pd.DataFrame:
    """Transform every GraphQL page in one Bronze payload into Silver rows."""
    pages = payload.get("pages")
    if not isinstance(pages, list):
        raise ValueError("Bronze payload must contain a 'pages' list")

    if not pages:
        return pd.DataFrame()

    page_frames = [
        _transform_page_to_silver(run_id, extracted_at, page)
        for page in pages
    ]
    return pd.concat(page_frames, ignore_index=True)



In [44]:
df = transform_bronze_to_silver(run_id="01", extracted_at=extracted_at, payload=data)

In [45]:
df.head()

,issue_id,issue_title,issue_url,issue_author,created_at,updated_at,state,reviewer,milestone,status,run_id,extracted_at,is_assigned,days_since_update,submission_age_days,current_milestone
0,I_kwDOShte388AAAABIkl5GA,[M2] <Froncoyz Verano> — <Philippine Regional ...,https://github.com/dataengineeringpilipinas/de...,Froncoyz,2026-07-13 11:50:03+08:00,2026-08-03 21:18:12+08:00,OPEN,[datadonotlieee],M2,Needs Improvement,01,2026-08-12 21:14:43+08:00,1,21,30,M2
1,I_kwDOShte388AAAABK9ne-w,[M2] <Naehum Dela Cruz> — <MassKara Festival G...,https://github.com/dataengineeringpilipinas/de...,Naehum-Dela-Cruz,2026-07-31 22:46:52+08:00,2026-08-04 18:26:17+08:00,CLOSED,[jlnrox],M2,Passed,01,2026-08-12 21:14:43+08:00,1,3,11,M3
2,I_kwDOShte388AAAABKgau4A,[M2] John Lopez — MaxiPay PH,https://github.com/dataengineeringpilipinas/de...,jatjatlopez,2026-07-28 23:06:08+08:00,2026-08-03 21:19:56+08:00,OPEN,[datadonotlieee],M2,Needs Improvement,01,2026-08-12 21:14:43+08:00,1,5,14,M2
3,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-23 00:17:50+08:00,2026-08-02 09:58:56+08:00,OPEN,[webzero13],M0,Passed,01,2026-08-12 21:14:43+08:00,1,40,50,M1
4,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10 20:18:57+08:00,2026-08-03 20:05:57+08:00,CLOSED,[nicholailim],M1,Passed,01,2026-08-12 21:14:43+08:00,1,23,33,M2


In [46]:
print(len(df))

121
